**使用指南：**

！！！使用前，请按需修改下面的环境配置

1.首次使用，先`Run all`运行一次，弹出widgets输入框（retry_condition_str-筛选条件，is_latest-默认true，是则根据businesskey取最新值，否则取所有值）
###### businesskey: market_code, brand_code, source_system_code, consumer_id

2.非首次使用，直接输入指定的筛选条件字符串`retry_condition_str`（示例：[ _TASK_ID = "xxx" and SRCC_CONSUMERID = "xxx"_ ]），回车后自动运行下面脚本

3.脚本会将筛选出的Consumer源数据自动发送到kafka目标topic（根据market溯源找到源topic）

In [0]:
## ===================== 环境配置 =======================
# Kafka broker 地址
kafka_brokers = "10.249.200.43:9092,10.249.200.42:9092,10.249.200.44:9092"
# database & bronze path
config_database = "catalog_southeastasia_mdm_share_prod.share_mdm_config"
silver_consumer_cleansed_database = "catalog_southeastasia_mdm_silver_prod.consumer_cleansed"
bronze_path_consumer = "abfss://bronze@saapseaprodacdgen2.dfs.core.windows.net/cdp_mdm_prod/mdm_raw_landing/consumerlist_raw"
## ======================================================

import time
from pyspark.sql.window import Window
from pyspark.sql import functions as F

def parse_bool(v):
    return str(v).strip().lower() == "true"

# 取数的条件字符串：用于在 cleansed 表中过滤需要重试的记录，例如 "task_id = 'xxx'"
dbutils.widgets.text("retry_condition_str", "")
condition_str = dbutils.widgets.get("retry_condition_str")

# 是否仅拉取每个消费者最新版本的数据（默认 true）。
# 为 true 时，会按 market_code + brand + source + consumerid 分组，取 srcc_sourcetimestamp 最新的一条。
dbutils.widgets.text("is_latest", "true")
is_latest = parse_bool(dbutils.widgets.get("is_latest"))

retry_task_id = str(int(time.time() * 1000))
print(f"----------------- retry_task_id: {retry_task_id} -----------------\n")
print(f"condition_str: [{condition_str}]")
print(f"is_latest: {is_latest}")
print(f"kafka brokers: {kafka_brokers}")

def convert_colum_type(originalDf, targetDf):
    result = originalDf
    originalSchema = originalDf.schema
    targetSchema = targetDf.schema
    # Step 1: 检查 original 中的列是否在 target 中
    for originalCol in originalSchema:
        originalType = originalCol.dataType
        originalName = originalCol.name
        tar_cols = [tar_col for tar_col in targetSchema if tar_col.name.upper() == originalName.upper()]
        if len(tar_cols) == 0:
            print(f"col:{originalName} is not in tarDF")
        else:
            tar_type = tar_cols[0].dataType
            if originalType != tar_type:
                print(f"origName={originalName} ; origType={originalType} ; tarType={tar_type}")
                result = result.withColumn(originalName, F.col(originalName).cast(tar_type))
    # Step 2: 【新增】检查 target 中的列是否在 original 中
    for targetCol in targetSchema:
        targetName = targetCol.name
        orig_cols = [orig_col for orig_col in originalSchema if orig_col.name.upper() == targetName.upper()]
        if len(orig_cols) == 0:
            print(f"col:{targetName} is not in orgDF")
    return result

def append_table(df, table):
    tmpDf = df
    tarDf = spark.table(f"{table}").where(F.lit(False))
    convert_colum_type(tmpDf, tarDf).write.format('delta').mode('append').option("mergeSchema", "true").saveAsTable(table)

def find_retry_candidate_df(condition_str, is_latest):
    """
    根据条件从 cleansed 层 (t_clean_consumer) 中查找需要重试的消费者候选记录。

    参数:
        condition_str (str): SQL WHERE 条件字符串，用于筛选目标记录。
        is_latest (bool): 为 True 时，仅保留每个消费者的最新版本（按 srcc_sourcetimestamp 降序）。

    返回:
        DataFrame: 包含 task_id, market_code, srcc_id 的去重结果。
    """
    # cleansed 层消费者表，包含清洗后的消费者数据
    clean_consumer_table = f"{silver_consumer_cleansed_database}.t_clean_consumer"

    # 先根据条件过滤并选取关联字段
    base_df = (
        spark.table(clean_consumer_table)
        .where(condition_str)
        .select(
            F.col("srcc_mrkt_code").alias("market_code"),
            F.col("srcc_id"),
            F.col("srcc_brnd_code"),
            F.col("srcc_srcs_code"),
            F.col("srcc_consumerid"),
            F.col("srcc_sourcetimestamp"),
        )
    )

    # 如果不需要取最新版本，直接返回去重的 market_code + srcc_id
    if not is_latest:
        return base_df.select("market_code", "srcc_id").distinct()

    # 按消费者业务键分组（market + brand + source + consumerid），
    # 按 srcc_sourcetimestamp 降序（nulls_last），时间相同则按 srcc_id 降序，取第一条
    latest_df = (
        base_df.withColumn(
            "rank_num",
            F.row_number().over(
                Window.partitionBy(
                    F.col("market_code"),
                    F.col("srcc_brnd_code"),
                    F.col("srcc_srcs_code"),
                    F.col("srcc_consumerid"),
                ).orderBy(
                    F.col("srcc_sourcetimestamp").desc_nulls_last(),
                    F.col("srcc_id").desc(),
                )
            ),
        )
        .filter(F.col("rank_num") == 1)  # 只保留每个分组内最新的记录
        .select("market_code", "srcc_id")
        .distinct()
    )
    return latest_df


def build_kafka_retry_df(retry_df):
    """
    根据重试候选记录，从 bronze 层构建需要重新发送到 Kafka 的消息 DataFrame。

    参数:
        retry_df (DataFrame): 包含 task_id, market_code, srcc_id 的候选记录。

    返回:
        DataFrame: 包含 market_code, task_id, srcc_id, kafka_topic, kafka_key, kafka_value 的 Kafka 消息。
                   其中 kafka_topic 和 kafka_value 来自 bronze 层的原始 Kafka 记录。
    """
    # 从 bronze 层读取原始消费者数据（存储了从 Kafka 接入的原始 payload）
    bronze_df = spark.read.format("delta").load(bronze_path_consumer)

    # 提取候选记录的关键字段，用于与 bronze 层做关联
    retry_keys_df = retry_df.select("market_code", "srcc_id").distinct()

    # 通过 task_id + market_code + srcc_id 关联 bronze 层原始数据，
    # 从 JSON payload 中提取 MarketCode，将 slndc_id 转为字符串以匹配 srcc_id
    kafka_df = (
        bronze_df.alias("b")
        .withColumn("retry_market_code", F.get_json_object("slndc_payload", "$.source.Consumer.SourceSystem.MarketCode"))
        .withColumn("retry_srcc_id", F.col("slndc_id").cast("string"))
        .join(
            retry_keys_df.alias("r"),
            (F.col("retry_market_code") == F.col("r.market_code"))
            & (F.col("retry_srcc_id") == F.col("r.srcc_id")),
            "inner",
        )
        .select(
            F.col("retry_market_code").alias("market_code"),
            F.col("retry_srcc_id").alias("srcc_id"),
            # 去掉 bronze 层 topic 名称末尾的 _Validated 后缀，得到实际要发送的目标 topic
            F.regexp_replace(F.col("b.slndc_kafka_topic").cast("string"), "_Validated$", "").alias("kafka_topic"),
            F.col("slndc_kafka_key").alias("kafka_key"),
            # 从原始 payload 中提取 $.source 字段作为最终发送的 Kafka value
            F.get_json_object(F.col("b.slndc_payload"), "$.source").alias("kafka_value"),
        )
        # 过滤掉 topic 或 value 为空的记录，确保 Kafka 消息有效
        .filter(F.col("kafka_topic").isNotNull() & (F.length(F.col("kafka_topic")) > 0))
        .filter(F.col("kafka_value").isNotNull() & (F.length(F.col("kafka_value")) > 0))
        # 去重：同一 task + market + srcc_id + topic + key + value 只发一次
        .dropDuplicates(["market_code", "srcc_id", "kafka_topic", "kafka_key", "kafka_value"])
    )
    return kafka_df


def send_retry_messages_to_kafka(kafka_df):
    """
    将重试消息发送到 Kafka。

    参数:
        kafka_df (DataFrame): 已构建好的 Kafka 消息 DataFrame，包含 kafka_topic, kafka_key, kafka_value。
    """
    # 使用 Spark Kafka DataSource 写入消息
    (
        kafka_df.select(
            F.col("kafka_key"),
            F.col("kafka_value").alias("value"),
            F.col("kafka_topic").alias("topic"),
        )
        .write.format("kafka")
        .option("kafka.bootstrap.servers", kafka_brokers)
        .option("kafka.request.timeout.ms", "15000")    # 请求超时 15 秒
        .option("kafka.max.block.ms", "20000")          # 发送阻塞最大 20 秒
        .option("kafka.delivery.timeout.ms", "30000")   # 投递超时 30 秒
        .option("kafka.retries", "0")                   # 不重试，失败即报错
        .mode("append")
        .save()
    )


def retry_for_standard(condition_str, is_latest):
    """
    标准重试主流程：根据条件找到源数据，构建 Kafka 消息后发送，并写入重试日志。

    整体逻辑：
      1. 校验参数；
      2. 从 cleansed 层查找候选记录；
      3. 关联 bronze 层构建 Kafka 消息 DataFrame；
      4. 发送到 market 映射的 Kafka topic；
      5. 将重试记录（含实际发送的 topic）写入 retry_config 表；
      6. 输出统计信息。

    参数:
        condition_str (str): 过滤条件，用于在 cleansed 表中筛选目标记录。
        is_latest (bool): 是否仅取每个消费者最新版本。
    """
    # 条件字符串不能为空，否则无法定位需要重试的数据
    if not condition_str or condition_str.strip() == "":
        print("[WARNING] condition_str must not be blank.")
        return

    # 重试配置表，用于记录重试历史
    retry_config_table = f"{config_database}.retry_config"

    # 步骤 1：从 cleansed 层查找候选记录 (task_id, market_code, srcc_id)
    candidate_df = find_retry_candidate_df(condition_str, is_latest)
    candidate_count = candidate_df.count()

    print(f"[INFO] retry candidate count: {candidate_count}")

    # 没有候选记录则直接结束
    if candidate_count == 0:
        return

    # 步骤 2：关联 bronze 层构建 Kafka 消息 DataFrame
    kafka_df = build_kafka_retry_df(candidate_df).cache()  # 缓存结果以供后续使用
    kafka_count = kafka_df.count()

    print(f"[INFO] kafka retry message count: {kafka_count}")

    # 没有构建出 Kafka 消息则直接结束
    if kafka_count == 0:
        return

    # 步骤 3：发送到 market 映射的 Kafka topic
    send_retry_messages_to_kafka(kafka_df)

    # 步骤 4：构造重试日志数据并写入 retry_config 表，retry_target 取实际发送的 topic
    retry_log_df = (
        kafka_df.select(
            "market_code",
            "srcc_id",
            F.col("kafka_topic").alias("retry_target"),
        )
        .withColumn("task_id", F.lit(retry_task_id))
        .withColumn("condition_str", F.lit(condition_str))
        .withColumn("retry_type", F.lit("standard"))
        .withColumn("retry_status", F.lit("SUCCESS"))
        .withColumn("create_time", F.current_timestamp())
        .withColumn("create_user", F.lit("support"))
    )
    append_table(retry_log_df, retry_config_table)

    kafka_df.unpersist()  # 释放缓存

# main
# 执行标准重试主流程：根据 condition_str 和 is_latest 找到源数据并发送到 Kafka
retry_for_standard(condition_str=condition_str, is_latest=is_latest)